## Concept 4 — Structured Output Agent

### What is it?
An agent whose final answer is a **typed Pydantic object** instead of a plain string.
Two stages: the agent executes tools autonomously (Stage 1), then a structured LLM
reformats the agent's answer into a schema (Stage 2).

### Why Structured Output?
```
Plain string answer:       "144 divided by 12 is 12."
  → to extract the number you'd need regex or parsing

Structured output:
  response.answer     = "144 divided by 12 is 12."
  response.tools_used = ["calculate"]
  response.confidence = "high"
  → directly usable in downstream code, databases, APIs
```

### Pipeline
```
Query
  → Stage 1: create_agent executes tools → raw answer string
  → Stage 2: structured_llm fills Pydantic schema
  → AgentResponse object with typed fields
```

### Limitation (what Concept 5 fixes)
Still stateless — each call forgets previous turns.
→ Concept 5 adds message history so the agent remembers across turns.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

llm = get_llm()
print(f'LLM ready | Tools: {[t.name for t in TOOLS]}')

### Step 1 — Define the Output Schema
Pydantic fields define exactly what the final response must contain.
The descriptions guide the LLM on what to fill in each field.

In [ ]:
class AgentResponse(BaseModel):
    answer:     str       = Field(description='Complete answer to the user query')
    tools_used: List[str] = Field(description='Names of tools called, e.g. ["calculate"]')
    confidence: str       = Field(description='Confidence level: high, medium, or low')
    reasoning:  str       = Field(description='One sentence: how did you arrive at this answer?')

print('Schema fields:', list(AgentResponse.model_fields.keys()))

### Step 2 — Stage 1: create_agent Executes Tools
The agent runs autonomously. We extract its final text answer and tool usage from messages.

In [ ]:
agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt='You are a helpful assistant. Use tools whenever appropriate.',
)

def run_agent_and_extract(query: str):
    result    = agent.invoke({'messages': [{'role': 'user', 'content': query}]})
    messages  = result['messages']
    raw_answer = messages[-1].content

    # Extract which tools were actually called from the message trace
    used = []
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for call in msg.tool_calls:
                used.append(call['name'])
                print(f'  [Agent called] {call["name"]}({call["args"]})')

    return raw_answer, used

raw, tools_used = run_agent_and_extract('What is 144 divided by 12?')
print(f'Raw agent answer: {raw}')
print(f'Tools used:       {tools_used}')

### Step 3 — Stage 2: Structured Synthesis
`with_structured_output` forces the LLM to fill every field of `AgentResponse`.

In [ ]:
structured_llm = llm.with_structured_output(AgentResponse)

synthesis_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Convert the agent answer into a structured response. Fill all fields accurately.'),
    ('human',
     'Query: {query}\n'
     'Agent answer: {raw_answer}\n'
     'Tools actually called: {tools_used}\n'
     'Fill the schema now.'),
])
synthesis_chain = synthesis_prompt | structured_llm

query = 'What is 144 divided by 12?'
raw, used = run_agent_and_extract(query)

structured = synthesis_chain.invoke({'query': query, 'raw_answer': raw, 'tools_used': str(used)})

print(f'\nType:       {type(structured).__name__}  ← Pydantic object, not a string')
print(f'answer:     {structured.answer}')
print(f'tools_used: {structured.tools_used}')
print(f'confidence: {structured.confidence}')
print(f'reasoning:  {structured.reasoning}')

### Step 4 — Compare: String vs Pydantic Output
See why structured output is better for real applications.

In [ ]:
q = 'What is the weather in Hyderabad?'
raw2, used2 = run_agent_and_extract(q)

print(f'String output:  "{raw2}"')
print(f'  → to get the temperature you need string parsing\n')

structured2 = synthesis_chain.invoke({'query': q, 'raw_answer': raw2, 'tools_used': str(used2)})
print(f'Pydantic output:')
print(f'  .answer     = {structured2.answer}')
print(f'  .tools_used = {structured2.tools_used}')
print(f'  .confidence = {structured2.confidence}')
print(f'  → fields accessed directly, no parsing needed')

### Full Function

In [ ]:
def structured_output_agent(query: str) -> str:
    raw, used = run_agent_and_extract(query)
    response  = synthesis_chain.invoke({'query': query, 'raw_answer': raw, 'tools_used': str(used)})
    return f'{response.answer}  [tools={response.tools_used}, confidence={response.confidence}]'

### Test — Standard Queries

In [ ]:
run_queries(structured_output_agent, label='Concept 4 — Structured Output Agent')